In [ ]:
# import json, os, pickle
# import numpy as np
# from tqdm.notebook import tqdm
# import pandas as pd

# import torch
# from torch.utils.data import DataLoader, TensorDataset
# import matplotlib.pyplot as plt
# import seaborn as sns
# import qiskit.circuit.random
# import torch, random
# from torch.utils.data import Dataset, DataLoader, TensorDataset
# from torch.optim.lr_scheduler import ReduceLROnPlateau
# import torch.nn as nn

# import numpy as np
# import json, os, pickle
# from tqdm import tqdm
# import pandas as pd

# import matplotlib.pyplot as plt
# import seaborn as sns

# from qiskit import QuantumCircuit

# import sys
# sys.path.append('../tutorials/')
# from mlp import encode_data, encode_data_v2_ecr

In [1]:
import os
import qiskit
from qiskit.circuit import QuantumCircuit
from qiskit.qpy import load
from typing import List
from tqdm import tqdm

In [23]:
def load_qpy_circuits_from_folder(folder_path: str) -> List[QuantumCircuit]:
    """
    Load all .qpy circuit files from a specified folder.

    Parameters:
        folder_path (str): The path to the folder containing .qpy files.

    Returns:
        List[QuantumCircuit]: A list of QuantumCircuit objects.
    """
    circuits = []

    # Ensure folder exists
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    # Loop through all files in the directory
    for filename in tqdm(os.listdir(folder_path)):
        if filename.endswith(".qpy"):
            qpy_path = os.path.join(folder_path, filename)
            try:
                with open(qpy_path, "rb") as f:
                    # A .qpy file can contain multiple circuits
                    circuits.extend(load(f))
            except:
                # print
                continue
        # break

    return circuits

In [24]:
folder = "../../../andrew/ExecutionResults/StoredCircuits/"
loaded_circuits = load_qpy_circuits_from_folder(folder)


100%|█████████████████████████████████████████████████████████| 7004/7004 [00:10<00:00, 647.88it/s]


In [25]:
print(f"Number of circuits loaded = {len(loaded_circuits)}")

Number of circuits loaded = 6559


In [26]:
import os
from qiskit import transpile
from qiskit.qpy import load
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeLimaV2
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit.result import Result
from qiskit.circuit import QuantumCircuit
from typing import List, Tuple

In [27]:
def simulate_circuits_with_fake_lima_v2(circuits: List[QuantumCircuit], shots: int = 1024) -> List[Tuple[Result, Result]]:
    results = []

    # Define Fake backend
    fake_backend = FakeFez()
    noisy_sim = AerSimulator.from_backend(fake_backend)
    ideal_sim = AerSimulator()

    for idx, qc in enumerate(circuits):
        print(f"\nSimulating circuit {idx + 1}/{len(circuits)}")

        # Transpile for both simulators
        tqc_noisy = transpile(qc, backend=noisy_sim)
        tqc_ideal = transpile(qc, backend=ideal_sim)

        # Run
        job_noisy = noisy_sim.run(tqc_noisy, shots=shots)
        job_ideal = ideal_sim.run(tqc_ideal, shots=shots)

        results.append((job_ideal.result(), job_noisy.result()))

    return results

In [28]:
results = simulate_circuits_with_fake_lima_v2(loaded_circuits)


Simulating circuit 1/6559

Simulating circuit 2/6559

Simulating circuit 3/6559

Simulating circuit 4/6559

Simulating circuit 5/6559

Simulating circuit 6/6559

Simulating circuit 7/6559


KeyboardInterrupt: 

In [29]:
results

[(Result(backend_name='aer_simulator', backend_version='0.17.1', job_id='da8a4241-cdaf-4fd2-bd62-e98bff0d7b62', success=True, results=[ExperimentResult(shots=1024, success=True, meas_level=2, data=ExperimentResultData(counts={'0x2cb': 1024}), header={'creg_sizes': [['meas', 10]], 'global_phase': 0.0, 'memory_slots': 10, 'n_qubits': 10, 'name': '5-qubit QFTAdder', 'qreg_sizes': [['q', 10]], 'metadata': {}}, status=DONE, seed_simulator=1214342026, metadata={'time_taken': 0.022683076, 'num_bind_params': 1, 'parallel_state_update': 24, 'parallel_shots': 1, 'required_memory_mb': 1, 'input_qubit_map': [[9, 9], [8, 8], [7, 7], [6, 6], [5, 5], [4, 4], [3, 3], [2, 2], [1, 1], [0, 0]], 'method': 'statevector', 'device': 'CPU', 'num_qubits': 10, 'sample_measure_time': 0.010974512, 'active_input_qubits': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'num_clbits': 10, 'remapped_qubits': False, 'runtime_parameter_bind': False, 'max_memory_mb': 31779, 'noise': 'ideal', 'measure_sampling': True, 'batched_shots_opti

In [15]:
from collections import Counter

def counts_to_z_expectation(counts: dict, n_qubits: int) -> list:
    """
    Compute expectation values ⟨Z⟩ for each qubit from measurement counts.

    Parameters:
        counts (dict): Dictionary of bitstring outcomes and their frequencies.
        n_qubits (int): Number of qubits (bitstring length)

    Returns:
        List[float]: ⟨Z⟩ expectation values per qubit
    """
    total_shots = sum(counts.values())
    expectations = [0.0] * n_qubits

    for bitstring_hex, count in counts.items():
        # Convert hex to binary with fixed width
        bitstring = bin(int(bitstring_hex, 16))[2:].zfill(n_qubits)
        bitstring = bitstring[::-1]  # Reverse to match Qiskit's qubit ordering

        for i in range(n_qubits):
            z = 1 if bitstring[i] == '0' else -1
            expectations[i] += z * count

    return [round(e / total_shots, 4) for e in expectations]

In [16]:
# Unpack your results
ideal_result, noisy_result = results[0]  # Replace `results` with your actual list

# Extract counts
ideal_counts = ideal_result.results[0].data.counts
noisy_counts = noisy_result.results[0].data.counts

# Compute expectations
n_qubits = 10
ideal_z = counts_to_z_expectation(ideal_counts, n_qubits)
noisy_z = counts_to_z_expectation(noisy_counts, n_qubits)

# Display
print("⟨Z⟩ values (ideal):", ideal_z)
print("⟨Z⟩ values (noisy):", noisy_z)


⟨Z⟩ values (ideal): [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
⟨Z⟩ values (noisy): [-0.8848, -0.8926, 0.9297, -0.9395, 0.9668, 0.8477, -0.7812, -0.8027, 0.7227, -0.6914]


In [38]:
import json

def simulate_and_store_z_expectations_json(qpy_folder: str, output_json_path: str, shots: int = 1024):
    """
    Loads QPY circuits, simulates them (ideal and noisy), computes ⟨Z⟩ expectations,
    and saves the results to a JSON file.

    Parameters:
        qpy_folder (str): Folder path containing .qpy files.
        output_json_path (str): Path to save the output .json dictionary.
        shots (int): Number of simulation shots.
    """
    def counts_to_z_expectation(counts: dict, n_qubits: int) -> List[float]:
        total_shots = sum(counts.values())
        expectations = [0.0] * n_qubits

        for bitstring_hex, count in counts.items():
            bitstring = bin(int(bitstring_hex, 16))[2:].zfill(n_qubits)[::-1]
            for i in range(n_qubits):
                z = 1 if bitstring[i] == '0' else -1
                expectations[i] += z * count

        return [round(e / total_shots, 4) for e in expectations]

    results_dict = {}
    fake_backend = FakeFez()
    noisy_sim = AerSimulator.from_backend(fake_backend)
    ideal_sim = AerSimulator()

    for file in os.listdir(qpy_folder):
        if not file.endswith(".qpy"):
            continue

        file_path = os.path.join(qpy_folder, file)
        try:
            with open(file_path, "rb") as f:
                circuits = load(f)
        except Exception as e:
            print(f"Error loading {file}: {e}")
            continue

        for idx, qc in enumerate(circuits):
            print(f"Simulating {file} (circuit {idx + 1})...")

            n_qubits = qc.num_qubits
            tqc_ideal = transpile(qc, backend=ideal_sim)
            tqc_noisy = transpile(qc, backend=noisy_sim)

            ideal_result = ideal_sim.run(tqc_ideal, shots=shots).result()
            noisy_result = noisy_sim.run(tqc_noisy, shots=shots).result()

            ideal_counts = ideal_result.results[0].data.counts
            noisy_counts = noisy_result.results[0].data.counts

            z_ideal = counts_to_z_expectation(ideal_counts, n_qubits)
            z_noisy = counts_to_z_expectation(noisy_counts, n_qubits)

            results_dict[file_path] = {
                'z_ideal': z_ideal,
                'z_noisy': z_noisy
            }
        break
    # Convert to JSON-safe format
    with open(output_json_path, "w") as json_out:
        json.dump(results_dict, json_out, indent=2)

    print(f"\n✅ Results saved to {output_json_path}")


In [39]:
simulate_and_store_z_expectations_json(
    qpy_folder="../../../andrew/ExecutionResults/StoredCircuits/",
    output_json_path="z_expectations.json",
    shots=1024
)


Simulating 9442e55b-2c38-4287-85c0-61af0dcc7dbc.qpy (circuit 1)...

✅ Results saved to z_expectations.json
